In [ ]:
import pandas as pd 
import numpy as np

In [136]:
book_ratings_raw = pd.read_csv('../data/BookCrossingThemes.csv', sep = ';')
book_ratings_raw.head()

,Book-Title,Book-Author,User-ID,ISBN,Book-Rating,Year-Of-Publication,Publisher,Location,Age,category,description,num_words,num_chars,cleaned_description,Theme
0,The Terminal Man,Michael Crichton,276964,345354621,10,1988,Ballantine Books,"villa ridge, missouri, usa",34.0,Fiction,Hearry Benson suffers from violent seizures. W...,54,335,hearry benson suffers violent seizure becomes ...,Psychological Thriller
1,The Chamber,John Grisham,276964,440220602,9,1995,Dell Publishing Company,"villa ridge, missouri, usa",34.0,American fiction,While the executioners prepare the gas chamber...,26,155,executioner prepare gas chamber protester gath...,Crime Thrillers and Detective Drama
2,The Girl Who Loved Tom Gordon : A Novel,Stephen King,276964,684867621,8,1999,Scribner,"villa ridge, missouri, usa",34.0,Fiction,A story of a nine year old who wanders off in ...,19,90,story nine year old wanders wilderness area he...,Spiritual and Self-Help
3,In the Dark,Richard Laymon,276964,843949163,8,2001,Leisure Books,"villa ridge, missouri, usa",34.0,California,A tale of suspense follows young librarian Jan...,44,250,tale suspense follows young librarian jane ker...,Vampire/Paranormal Fantasy
4,Tailchaser's Song,Tad Williams,276964,886773741,7,1994,Daw Books,"villa ridge, missouri, usa",34.0,Fiction,"Fritti Tailchaser, a ginger tomcat of courage ...",31,199,fritti tailchaser ginger tomcat courage curios...,Cozy Mysteries


In [ ]:
book_ratings_raw.info()

In [ ]:
# Shows all rows where the category string starts with a bracket
list_like_examples = book_ratings_raw[book_ratings_raw['category'].astype(str).str.startswith('[')]
print(list_like_examples['category'].head(10))

In [ ]:
sample_val = book_ratings_raw['category'].iloc[0]
print(f"Value: {sample_val}")
print(f"Data Type: {type(sample_val)}") 
# If it says <class 'str'>, it's just text. 
# If it says <class 'list'>, it's a real Python list.

In [137]:
book_ratings = book_ratings_raw.copy()

# This regex removes [ , ] and ' characters all at once
book_ratings['category_clean'] = book_ratings_raw['category'].astype(str).str.replace(r"[\[\]']", "", regex=True).str.strip()

# Now check the unique values again
print(book_ratings['category_clean'].unique()[:10])

['Fiction' 'American fiction' 'California' 'nan' 'Humor'
 'Adventure stories' 'Davenport, Lucas (Fictitious character)'
 'Diary fiction' 'Language Arts & Disciplines'
 'Fiction,Romance,Cultural,African American,Womens Fiction,Chick Lit,Contemporary,Novels,Adult Fiction,Romance,Contemporary Romance,Womens,American,African American Literature']


In [138]:
# Split by comma and take the first item
book_ratings['temp_category'] = book_ratings['category_clean'].str.split(',').str[0]

print(book_ratings['temp_category'].value_counts().head(10))

temp_category
Fiction                      48865
nan                           3084
Juvenile Fiction              2882
Biography & Autobiography     2481
Humor                          836
American fiction               701
Bildungsromans                 375
Business & Economics           346
Family & Relationships         328
Adventure stories              307
Name: count, dtype: int64


In [ ]:
# 1. Create a filter for rows where primary_genre is missing
missing_mask = book_ratings['temp_category'].isna() | (book_ratings['temp_category'] == 'nan')

# 2. Look at the Theme values for those specific rows
fill_preview = book_ratings.loc[missing_mask, 'Theme']

# 3. Show the most common Themes that will be 'backfilling' your genres
print("Most frequent Themes being added to fill NaNs:")
print(fill_preview.value_counts().head(10))

# 4. (Optional) See 5 random examples of the actual rows
print("\nExamples of rows to be filled:")
print(book_ratings.loc[missing_mask, ['Book-Title', 'Theme', 'temp_category']].head())


In [ ]:
# Get unique values as sets
themes_unique = set(book_ratings['Theme'].astype(str).unique())
genres_unique = set(book_ratings['temp_category'].astype(str).unique())

# Find the intersection (values present in both)
overlap = themes_unique.intersection(genres_unique)

print(f"Number of overlapping values: {len(overlap)}")
print(f"Sample of shared values: {list(overlap)[:10]}")

In [139]:
# Convert both to lowercase sets to see if they match conceptually
themes_lower = set(book_ratings['Theme'].astype(str).str.lower().unique())
genres_lower = set(book_ratings['temp_category'].astype(str).str.lower().unique())

hidden_overlap = themes_lower.intersection(genres_lower)
print(f"Conceptual matches (case-insensitive): {len(hidden_overlap)}")

Conceptual matches (case-insensitive): 0


In [ ]:
# 1. Standardize 'nan' strings to actual nulls
book_ratings['temp_category'] = book_ratings['temp_category'].replace('nan', np.nan)

# 2. Fill missing genres with Theme
book_ratings['primary_category'] = book_ratings['temp_category'].fillna(book_ratings['Theme'])

# 3. Check the new distribution
print(book_ratings['primary_category'].value_counts().head(15))

In [ ]:
# Group by Theme and get the top 3 categories for each
top_categories = (
    book_ratings.groupby('Theme')['category']
    .apply(lambda x: x.value_counts().head(3))
    .reset_index(name='count')
)

print(top_categories)

In [ ]:
# 1. Count the number of duplicate user-item pairs. This will confirm if there are any duplicate reviews for the same book by the same user
duplicate_count = book_ratings.duplicated(subset=['User-ID', 'ISBN']).sum()
print(f"Number of duplicate (User, ISBN) pairs: {duplicate_count}")

In [ ]:
# Count unique ISBNs per Book-Title
isbn_counts = book_ratings.groupby('Book-Title')['ISBN'].nunique().sort_values(ascending=False)

print(f"Titles with multiple ISBNs: {len(isbn_counts[isbn_counts > 1])}")
print("\nTop 10 titles with the most ISBN aliases:")
print(isbn_counts.head(10))

In [ ]:
# Create a mapping of Title -> First ISBN found
title_to_isbn = book_ratings.groupby('Book-Title')['ISBN'].first().to_dict()

# Map all rows to use the 'Master ISBN' for their title
book_ratings['Master-ISBN'] = book_ratings['Book-Title'].map(title_to_isbn)

# Check the new count (should be 1 ISBN per Title)
new_counts = book_ratings.groupby('Book-Title')['Master-ISBN'].nunique()
print(f"Unique Master-ISBNs per title: {new_counts.max()}")

In [ ]:
duplicate_count = book_ratings.duplicated(subset=['User-ID', 'Master-ISBN']).sum()
print(f"Number of duplicate (User, ISBN) pairs: {duplicate_count}")

In [ ]:
# Drop duplicates based on User-ID and the new Master-ISBN
book_ratings = book_ratings.drop_duplicates(subset=['User-ID', 'Master-ISBN'], keep='first')

In [ ]:
book_ratings['User-ID'].dtype

In [ ]:
book_ratings['Master-ISBN'].dtype

In [ ]:
# Force every value in Master-ISBN to be a string and strip any accidental whitespace
book_ratings['Master-ISBN'] = book_ratings['Master-ISBN'].astype(str).str.strip()

In [ ]:
# Check if 0 is the most common rating by a large margin, in which case it's very likely that the 0 is implicity (meaning missing data, rather than that its a bad review)

# Check the frequency of each rating
print(book_ratings['Book-Rating'].value_counts().sort_index())

# Calculate what percentage of your data is '0'
zero_pct = (book_ratings['Book-Rating'] == 0).mean() * 100
print(f"\nPercentage of '0' ratings: {zero_pct:.2f}%")

In [ ]:
user_counts = book_ratings['User-ID'].value_counts()

print("User Rating Stats:")
print(user_counts.describe())

# How many users have only 1 rating?
ones = (user_counts == 1).sum()
print(f"\nUsers with only 1 rating: {ones} ({ones/len(user_counts)*100:.1f}%)")

# How many users have only 2 rating?
ones = (user_counts == 2).sum()
print(f"\nUsers with 2 ratings: {ones} ({ones/len(user_counts)*100:.1f}%)")

# How many users have only 3 rating?
ones = (user_counts == 3).sum()
print(f"\nUsers with 3 rating: {ones} ({ones/len(user_counts)*100:.1f}%)")

In [ ]:
import html

# Columns to clean
text_cols = ['Book-Title', 'Book-Author', 'Publisher']

for col in text_cols:
    # 1. Ensure the column is string type
    # 2. Unescape HTML entities (e.g., &amp; -> &)
    # 3. Trim leading/trailing whitespace
    book_ratings[col] = (
        book_ratings[col]
        .astype(str)
        .apply(html.unescape)
        .str.strip()
    )

print("Text columns cleaned: HTML entities decoded and whitespace trimmed.")

In [ ]:
# Apply Title Case to unify capitalization
for col in ['Book-Title', 'Book-Author', 'Publisher']:
    book_ratings[col] = book_ratings[col].str.title()

print("Columns converted to Title Case.")

In [ ]:
# Identify all rows that have a duplicate (User-ID, Book-Title) pair
duplicates_view = book_ratings[book_ratings.duplicated(subset=['User-ID', 'Book-Title'], keep=False)]

# Sort them so the pairs are next to each other
print(duplicates_view.sort_values(by=['User-ID', 'Book-Title']).head(10))

In [ ]:
# Group by User-ID and Book-Title
# Take the mean of 'Book-Rating' and keep the first occurrence for everything else
aggregation_rules = {col: 'first' for col in book_ratings.columns if col not in ['User-ID', 'Book-Title', 'Book-Rating']}
aggregation_rules['Book-Rating'] = 'mean'

book_ratings = book_ratings.groupby(['User-ID', 'Book-Title'], as_index=False).agg(aggregation_rules)

# Optional: Round the rating if you want to keep them as integers
book_ratings['Book-Rating'] = book_ratings['Book-Rating'].round().astype(int)

print("Duplicates merged using the mean rating.")

In [ ]:
book_ratings.head()

In [ ]:
import datetime

# 1. Define the range of valid years
# Any year above 'current_year + 1' is likely an error (unless it's a pre-order)
current_year = datetime.datetime.now().year

# 2. Identify invalid rows
# Checking for 0, negative numbers, or future years
invalid_years = book_ratings[
    (book_ratings['Year-Of-Publication'] <= 0)  | 
    (book_ratings['Year-Of-Publication'] > current_year + 1)
]

# 3. View the results
print(f"Total invalid years found: {len(invalid_years)}")
if not invalid_years.empty:
    print(invalid_years[['Book-Title', 'Year-Of-Publication']].value_counts().head(10))

In [ ]:
book_ratings['Year-Of-Publication'] = book_ratings['Year-Of-Publication'].replace(0, np.nan)

# 2. Fill NaNs with the median year for that author
book_ratings['Year-Of-Publication'] = book_ratings.groupby('Book-Author')['Year-Of-Publication'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
# 1. Identify the authors who have a year of 0
authors_with_nan = book_ratings[book_ratings['Year-Of-Publication'].isna()]['Book-Author'].unique()

# 2. Check how many books those specific authors have in the ENTIRE dataset
author_counts = book_ratings[book_ratings['Book-Author'].isin(authors_with_nan)].groupby('Book-Author').size()

# 3. Filter for authors who have MORE than just the one book with year 0
authors_with_data = author_counts[author_counts > 1]

print(f"Total authors with at least one Year-0 book: {len(authors_with_nan)}")
print(f"Authors you can successfully 'fix' using their other books: {len(authors_with_data)}")
print(f"Authors with NO other data (will need the global median): {len(authors_with_nan) - len(authors_with_data)}")

In [ ]:
# 1. Fill NaNs using the median year of each author's other books
book_ratings['Year-Of-Publication'] = book_ratings.groupby('Book-Author')['Year-Of-Publication'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Final check: ensure there are no NaNs left
remaining_nans = book_ratings['Year-Of-Publication'].isna().sum()
print(f"Remaining NaNs in Year-Of-Publication: {remaining_nans}")

In [ ]:
# 1. Fill any remaining NaNs with the global median of the entire column
global_median = book_ratings['Year-Of-Publication'].median()
book_ratings['Year-Of-Publication'] = book_ratings['Year-Of-Publication'].fillna(global_median)

# 2. Now it is safe to convert to integer
book_ratings['Year-Of-Publication'] = book_ratings['Year-Of-Publication'].astype(int)

print(f"Remaining NaNs: {book_ratings['Year-Of-Publication'].isna().sum()}")
print(f"New Column Type: {book_ratings['Year-Of-Publication'].dtype}")

In [ ]:
import ast
import re

In [ ]:
def normalize_to_title_case(cat):
    # Handle empty, null, or 'nan'
    if pd.isna(cat) or str(cat).lower() == 'nan' or str(cat).strip() == '':
        return []
    
    val = str(cat).strip()
    
    # 1. Handle string-represented lists like "['science fiction', 'DRAMA']"
    if val.startswith('[') and val.endswith(']'):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                # .title() turns "science fiction" into "Science Fiction"
                return [str(item).strip().title() for item in parsed if item]
        except (ValueError, SyntaxError):
            # Fallback for messy strings that look like lists
            val = re.sub(r"[\[\]\'\"]", "", val)
    
    # 2. Split by commas and title-case each element
    tags = [t.strip().title() for t in re.split(r',', val) if t.strip()]
    return tags

# Apply the normalization
book_ratings['category_list'] = book_ratings['category'].apply(normalize_to_title_case)

# Display a sample to verify
print(book_ratings[['category', 'category_list']].head())

In [ ]:
# Count rows where the list length is greater than 1
multi_tag_count = book_ratings['category_list'].apply(lambda x: len(x) > 1).sum()

# Calculate the percentage for context
total_rows = len(book_ratings)
percentage = (multi_tag_count / total_rows) * 100

print(f"Rows with multiple categories: {multi_tag_count}")
print(f"Percentage of dataset: {percentage:.2f}%")

In [ ]:
# 1. Count exact nulls
null_age_count = book_ratings['Age'].isna().sum()

# 2. Check for unrealistic outliers (common in this specific book dataset)
# Ages under 5 or over 100 are usually data entry errors.
outliers = book_ratings[(book_ratings['Age'] < 5) | (book_ratings['Age'] > 100)]

print(f"Total Nulls in Age: {null_age_count}")
print(f"Total Outliers (<5 or >100): {len(outliers)}")

In [ ]:
# Replace outliers with NaN so they can be filled by the median later
book_ratings.loc[(book_ratings['Age'] < 5) | (book_ratings['Age'] > 100), 'Age'] = np.nan

In [ ]:
# Group by Location and calculate median, standard deviation, and count
location_stats = book_ratings.groupby('Location')['Age'].agg(['median', 'std', 'count'])

# Filter for locations with enough data to be meaningful (e.g., > 10 users)
reliable_locations = location_stats[location_stats['count'] > 10].sort_values('std')

print("Locations with the most 'consistent' ages (Low Std Dev):")
print(reliable_locations.head(10))

print("\nLocations with the most 'diverse' ages (High Std Dev):")
print(reliable_locations.tail(10))

# Calculate the average standard deviation across all reliable locations
print(f"\nAverage variation (Std Dev) within locations: {reliable_locations['std'].mean():.2f} years")
print(f"\nAverage variation (Std Dev) across all rows: {location_stats['std'].mean():.2f} years")


In [ ]:
# 1. Fill NaNs using the median age for each specific Location
book_ratings['Age'] = book_ratings.groupby('Location')['Age'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Safety Fallback: For users in locations with NO age data at all, use the global median
global_median = book_ratings['Age'].median()
book_ratings['Age'] = book_ratings['Age'].fillna(global_median)

# 3. Final cleanup: Round to integer (ages aren't decimals)
book_ratings['Age'] = book_ratings['Age'].round().astype(int)

# 4. Verify no NaNs remain
print(f"Remaining Age NaNs: {book_ratings['Age'].isna().sum()}")

In [ ]:
user_counts = book_ratings['User-ID'].value_counts()

print("Top 10 Power Users (by number of ratings):")
print(user_counts.head(10))

# Calculate what percentage of the total data the top 1% of users represent
top_1_percent_threshold = int(len(user_counts) * 0.01)
top_users_share = user_counts.head(top_1_percent_threshold).sum() / len(book_ratings) * 100

print(f"\nThe top 1% of users account for {top_users_share:.2f}% of all ratings.")


In [ ]:
import os 

In [ ]:
# Define the path to the data folder
output_path = os.path.join('..', 'data', 'BookCrossingThemes_Updated.csv')

# Save the dataframe
book_ratings.to_csv(output_path, index=False)

print(f"File successfully saved to: {output_path}")